# 03 — Docker, Konfigurasi, dan Deploy ke Hugging Face Spaces

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/riki-profile/ai-engineer-journey/blob/main/12-deployment/03_docker_dan_deploy.ipynb)

API di notebook 01 berjalan di komputer kita. Agar berjalan **sama persis** di server mana pun (laptop rekan kerja, server kantor, cloud), aplikasi dikemas dalam **container**. Notebook ini membahas:

1. **Docker**: image, container, dan Dockerfile API kita.
2. **Konfigurasi dan rahasia** (API key) yang aman.
3. **Checklist produksi.**
4. **Deploy demo Gradio ke Hugging Face Spaces**, gratis dan bisa diakses publik.

> Docker tidak tersedia di Colab. Bagian Docker bisa dijalankan di komputer lokal yang sudah terpasang Docker, atau cukup dibaca: CI repo ini membangun dan menguji image yang sama secara otomatis di setiap pull request.

## 0. File Aplikasi dan Artefak

In [ ]:
%pip install -q fastapi uvicorn httpx gradio onnx onnxscript onnxruntime pandas huggingface_hub python-dotenv

In [ ]:
import os
import sys
import urllib.request
from pathlib import Path

QUICK_RUN = os.environ.get("QUICK_RUN") == "1"
REPO_RAW = "https://raw.githubusercontent.com/riki-profile/ai-engineer-journey/main/12-deployment/app"
FILE_APP = ["model.py", "latih.py", "main.py", "ui_gradio.py", "requirements.txt", "Dockerfile", ".dockerignore",
            "tests/test_api.py"]
APP = Path("app")
for f in FILE_APP:
    if not (APP / f).exists():
        (APP / f).parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(f"{REPO_RAW}/{f}", APP / f)

sys.path.insert(0, str(APP.resolve()))                         # agar `import model`, `import main` bisa dipakai
ARTEFAK = (APP / "artefak").resolve()
os.environ["FOLDER_MODEL"] = str(ARTEFAK)                      # konfigurasi API lewat environment variable
print(f"Folder aplikasi: {APP.resolve()} | QUICK_RUN: {QUICK_RUN}")

In [ ]:
from latih import latih

if not (ARTEFAK / "model.onnx").exists():                      # latih dulu jika belum ada (±20 detik di CPU)
    print(latih(ARTEFAK, epoch=5 if QUICK_RUN else 40))
print(sorted(f.name for f in ARTEFAK.iterdir()))

## 1. Docker: Image dan Container

Masalah klasik: "di komputer saya jalan kok". Versi Python, library, dan sistem operasi berbeda di setiap mesin. **Docker** mengemas aplikasi **beserta seluruh lingkungannya** menjadi satu **image**:

| Istilah | Analogi | Contoh |
|---|---|---|
| **Dockerfile** | Resep | Daftar langkah membangun image |
| **Image** | Kue beku yang siap dipanaskan | `api-sentimen:1.0`, bisa dikirim ke registry (Docker Hub, GHCR) |
| **Container** | Kue yang sedang disajikan | Proses yang berjalan dari sebuah image; bisa ada banyak dari satu image |

Image tersusun dari **layer**: setiap instruksi di Dockerfile menjadi satu layer yang di-cache. Urutan instruksi penting: langkah yang jarang berubah (instal library) diletakkan sebelum langkah yang sering berubah (kode), agar build ulang cepat.

Dockerfile API kita ([`app/Dockerfile`](app/Dockerfile)):

In [ ]:
print((APP / "Dockerfile").read_text())
print("--- .dockerignore ---")
print((APP / ".dockerignore").read_text())
print("--- requirements.txt (dependensi serving) ---")
print((APP / "requirements.txt").read_text())

Keputusan desain di Dockerfile tersebut:
- **Base image `python:3.11-slim`**: kecil, tetapi tetap berbasis Debian sehingga kompatibel dengan wheel Python.
- **Dependensi serving terpisah dari training**: image hanya berisi FastAPI, uvicorn, ONNX Runtime, dan NumPy, **tanpa PyTorch**. Hasilnya image ±400 MB, sedangkan PyTorch saja bisa lebih dari 1–2 GB. Image kecil lebih cepat di-download, di-scale, dan permukaan serangannya lebih kecil.
- **`requirements.txt` disalin dan diinstal sebelum kode**, agar layer instalasi tetap di-cache saat kode berubah.
- **Hanya artefak yang dibutuhkan** (`model.onnx`, `vocab.json`, `config.json`) yang disalin. `.dockerignore` mencegah file yang tidak perlu (misalnya `.env`, `model.pt`) ikut masuk.
- **User non-root**: jika aplikasi dibobol, penyerang tidak langsung mendapat akses root di container.
- **`HEALTHCHECK`**: Docker memeriksa `/health` secara berkala dan menandai container yang tidak sehat.

Perintah untuk membangun dan menjalankan (dari folder `12-deployment/app/`, setelah `python latih.py`):

```bash
docker build -t api-sentimen .
docker run -d --name api -p 8000:8000 --env-file .env api-sentimen   # .env berisi API_KEY=...
curl http://localhost:8000/health
curl -X POST http://localhost:8000/prediksi -H "Content-Type: application/json" \
     -H "X-API-Key: $API_KEY" -d '{"teks": "barangnya bagus"}'
docker logs api          # log permintaan dan latensi
docker stop api
```

Sel berikut menjalankan langkah yang sama **jika** Docker tersedia (misalnya Jupyter di komputer lokal). Di Colab dan saat pengujian otomatis, sel ini dilewati.

In [ ]:
import secrets
import shutil
import subprocess
import time

import httpx

docker_ada = shutil.which("docker") is not None and subprocess.run(["docker", "info"], capture_output=True).returncode == 0
if not docker_ada or QUICK_RUN:
    print("Docker tidak tersedia (atau QUICK_RUN): lewati. Jalankan perintah di atas di komputer dengan Docker.")
else:
    kunci = secrets.token_urlsafe(24)        # kunci acak hanya untuk uji lokal ini
    subprocess.run(["docker", "build", "-q", "-t", "api-sentimen", "."], cwd=APP, check=True)
    subprocess.run(["docker", "rm", "-f", "api-uji"], capture_output=True)
    subprocess.run(["docker", "run", "-d", "--name", "api-uji", "-p", "8000:8000", "-e", f"API_KEY={kunci}", "api-sentimen"],
                   check=True, capture_output=True)
    try:
        for _ in range(30):
            try:
                print("health:", httpx.get("http://127.0.0.1:8000/health", trust_env=False).json())
                break
            except httpx.TransportError:
                time.sleep(1)
        r = httpx.post("http://127.0.0.1:8000/prediksi", json={"teks": "barangnya bagus"},
                       headers={"X-API-Key": kunci}, trust_env=False)
        print("prediksi:", r.status_code, r.json())
        ukuran = subprocess.run(["docker", "images", "api-sentimen", "--format", "{{.Size}}"], capture_output=True, text=True)
        print("ukuran image:", ukuran.stdout.strip())
    finally:
        subprocess.run(["docker", "rm", "-f", "api-uji"], capture_output=True)

## 2. Konfigurasi dan Rahasia

Prinsip dari metodologi *Twelve-Factor App*: **konfigurasi disimpan di environment, bukan di kode**. Image yang sama bisa dipakai untuk pengembangan, staging, dan produksi; yang berbeda hanya environment variable-nya (`API_KEY`, `FOLDER_MODEL`, `MAKS_BATCH` di `main.py`).

Aturan untuk **rahasia** (API key, password database, token):
- Jangan pernah menulisnya di kode, Dockerfile (`ENV API_KEY=...`), atau menyalin `.env` ke image. Siapa pun yang punya image bisa membacanya (misalnya lewat `docker history` atau mengekstrak layer).
- Berikan saat container dijalankan: `-e`, `--env-file`, atau *secret manager* platform (GitHub Actions secrets, Hugging Face Spaces secrets, Google Secret Manager, dan lain-lain).
- Buat kunci yang panjang dan acak, misalnya dengan `secrets.token_urlsafe`. Contoh kunci baru (buat sendiri, simpan di `.env`, jangan di-commit):

In [ ]:
import secrets

print(secrets.token_urlsafe(32))

## 3. Checklist Produksi

| Area | Pertanyaan | Di contoh kita |
|---|---|---|
| Kesehatan | Ada endpoint health check? | `/health` + `HEALTHCHECK` Docker |
| Validasi | Input dibatasi (panjang, jumlah)? | Pydantic: 1–2000 karakter, batch ≤ 32 |
| Keamanan | Autentikasi? Rahasia di environment? Non-root? | API key dari `API_KEY`, user `appuser` |
| Observabilitas | Log permintaan, latensi, error? | Middleware mencatat latensi; lanjut di modul 13 |
| Reproduksibilitas | Versi library dipin? Versi model tercatat? | Versi model di `config.json`; pin versi library adalah Latihan 1 |
| Skala | Bisa dijalankan beberapa replika? | Stateless: tidak ada state di memori selain model |
| Pengujian | Tes otomatis sebelum deploy? | `pytest` + build & uji image Docker di CI |
| Jaringan | HTTPS, CORS, rate limiting? | Biasanya di reverse proxy / platform (Latihan) |

## 4. Deploy Demo ke Hugging Face Spaces

**Hugging Face Spaces** menyediakan hosting gratis untuk demo ML (Gradio, Streamlit, atau Docker). Sebuah Space adalah repository Git berisi aplikasi; setiap kali di-push, Space dibangun ulang otomatis. Cocok untuk **portofolio**.

Kita siapkan folder `space/` berisi:
- `app.py`: salinan `ui_gradio.py` (Spaces menjalankan `app.py`).
- `model.py` dan artefak ONNX (tanpa PyTorch, sehingga build cepat dan ringan).
- `requirements.txt`: dependensi untuk Space.
- `README.md` dengan **metadata YAML** yang dibaca Spaces (SDK, file utama, lisensi).

In [ ]:
SPACE = Path("space")
(SPACE / "artefak").mkdir(parents=True, exist_ok=True)
shutil.copy(APP / "ui_gradio.py", SPACE / "app.py")
shutil.copy(APP / "model.py", SPACE / "model.py")
for f in ["config.json", "vocab.json", "model.onnx"]:
    shutil.copy(ARTEFAK / f, SPACE / "artefak" / f)
(SPACE / "requirements.txt").write_text("onnxruntime\nnumpy\n")      # gradio sudah disediakan oleh SDK Space
(SPACE / "README.md").write_text("""---
title: Sentimen Bahasa Indonesia
emoji: 💬
colorFrom: blue
colorTo: green
sdk: gradio
app_file: app.py
license: agpl-3.0
---

Demo analisis sentimen bahasa Indonesia (model embedding + MLP kecil, disajikan dengan ONNX Runtime).
Dilatih pada NusaX-Senti (IndoNLP/nusax, lisensi CC-BY-SA). Bagian dari modul 12 repo ai-engineer-journey.
""")
for f in sorted(SPACE.rglob("*")):
    if f.is_file():
        print(f"{str(f.relative_to(SPACE)):22} {f.stat().st_size / 1e3:8.1f} KB")

Sebelum meng-upload, **uji dulu secara lokal** bahwa `app.py` di folder `space/` bisa dimuat dan memberi prediksi:

In [ ]:
import runpy

os.environ["FOLDER_MODEL"] = str((SPACE / "artefak").resolve())
modul_space = runpy.run_path(str(SPACE / "app.py"), run_name="uji_space")     # bukan __main__: UI tidak diluncurkan
print(modul_space["analisis"]("Pelayanannya ramah dan cepat!"))
os.environ["FOLDER_MODEL"] = str(ARTEFAK)

**Upload ke Spaces.** Butuh akun Hugging Face dan **token** dengan izin *write* (Settings → Access Tokens). Simpan token di Colab Secrets bernama `HF_TOKEN` atau di file `.env`, **jangan di kode**.

Upload adalah aksi publik: Space bisa dilihat siapa saja. Karena itu sel berikut hanya berjalan jika kamu sengaja mengubah `UNGGAH_KE_SPACES = True`.

In [ ]:
from dotenv import find_dotenv, load_dotenv

UNGGAH_KE_SPACES = False        # ubah ke True jika ingin meng-upload demo ke Space publik milikmu
NAMA_SPACE = "sentimen-indonesia"


def ambil_hf_token():
    load_dotenv(find_dotenv(usecwd=True))
    if os.getenv("HF_TOKEN"):
        return os.getenv("HF_TOKEN")
    try:
        from google.colab import userdata
        return userdata.get("HF_TOKEN")
    except Exception:
        return None


if not UNGGAH_KE_SPACES or QUICK_RUN:
    print("Upload dilewati (UNGGAH_KE_SPACES = False). Folder space/ sudah siap.")
elif not (token := ambil_hf_token()):
    print("HF_TOKEN belum diatur: tambahkan di Colab Secrets atau .env")
else:
    from huggingface_hub import HfApi

    api = HfApi(token=token)
    repo_id = f"{api.whoami()['name']}/{NAMA_SPACE}"
    api.create_repo(repo_id, repo_type="space", space_sdk="gradio", exist_ok=True)
    api.upload_folder(folder_path=str(SPACE), repo_id=repo_id, repo_type="space")
    print(f"✅ Space dibangun di https://huggingface.co/spaces/{repo_id} (butuh 1–3 menit)")

Setelah upload, Space membangun lingkungan dan menjalankan `app.py`. Pantau log build di tab *Logs* Space. Untuk memperbarui, jalankan ulang sel upload; Spaces menyimpan riwayatnya seperti Git.

### Pilihan deploy lain

| Platform | Cocok untuk | Catatan |
|---|---|---|
| **HF Spaces (Gradio/Streamlit)** | Demo, portofolio | Gratis (CPU), tidur jika lama tidak dipakai |
| **HF Spaces (Docker SDK)** | API FastAPI kita | Pakai Dockerfile yang sama; set `API_KEY` di *Secrets* Space |
| **Google Cloud Run / AWS App Runner / Azure Container Apps** | API produksi dari image Docker | Bayar per pemakaian, bisa scale ke nol, HTTPS otomatis |
| **Kubernetes** | Banyak layanan, tim besar | Kompleks; biasanya belum perlu di awal |
| **Serverless GPU / endpoint terkelola** | Model besar butuh GPU | Lebih mahal; perhatikan *cold start* |

Contoh Cloud Run (setelah memasang `gcloud` dan login): `gcloud run deploy api-sentimen --source . --set-secrets API_KEY=api-key:latest`. Rahasia diambil dari Secret Manager, bukan ditulis di perintah.

Setelah model berjalan di produksi, pekerjaan belum selesai: data berubah, performa bisa menurun, versi model berganti. Itu topik **MLOps** di modul 13.

## Ringkasan

- **Docker** mengemas aplikasi + lingkungannya menjadi image yang berjalan sama di mana saja. Susun layer dari yang jarang berubah ke yang sering berubah.
- Image serving sebaiknya **ramping**: dependensi serving saja (ONNX Runtime, bukan PyTorch), artefak yang diperlukan saja, user non-root, health check.
- **Konfigurasi dan rahasia lewat environment**, tidak pernah di kode atau image.
- Pakai **checklist produksi**: health, validasi, keamanan, observabilitas, reproduksibilitas, skala, pengujian.
- **Hugging Face Spaces** untuk demo publik gratis; Cloud Run dan sejenisnya untuk API produksi dari image Docker.

## Latihan

1. **Pin versi.** Buat `requirements.lock` dengan versi persis semua dependensi serving (misalnya lewat `pip freeze` di container yang berjalan), ubah Dockerfile untuk memakainya, lalu build ulang. Mengapa build yang reproducible penting? Apa kekurangannya (petunjuk: pembaruan keamanan)?
2. **Multi-stage build.** Ubah Dockerfile menjadi *multi-stage*: stage pertama memasang PyTorch dan menjalankan `latih.py` untuk menghasilkan artefak, stage kedua hanya menyalin artefak ke image serving yang ramping. Bandingkan ukuran image akhir dan waktu build.
3. **Deploy API ke Spaces (Docker SDK).** Buat Space baru dengan `sdk: docker` dan `app_port: 8000` di README, upload isi folder `app/` beserta artefak, lalu set `API_KEY` di *Settings → Secrets* Space. Panggil API publikmu dengan `httpx` dari notebook dan ukur latensinya dari Indonesia. Bandingkan dengan latensi lokal di notebook 01.